# model_learn — train the `small` model on a Colab GPU

This notebook trains the ~14M-parameter `small` config on TinyStories using the
**exact same** `src/slm` code that runs the local `toy` job — only the config and
compute change. See `notebooks/README.md` for step-by-step instructions.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.

You will download 3 files at the end into your local `checkpoints/`:
`small.pt`, `small_tok.json`, `small_loss.png`.


## 1. Setup — clone the repo and install deps

We install only our two direct deps (`datasets`, `tokenizers`) rather than the
full `requirements.txt`. Colab already ships torch/numpy/pandas/matplotlib in a
co-tuned set; force-installing the full pinned tree would clobber them and spew
resolver conflicts (and could need a runtime restart). Minimal install avoids all that.


In [ ]:
# Pre-filled to the pushed repo; change only if you forked/renamed it.
REPO_URL = "https://github.com/vppillai/model_learn.git"

!git clone $REPO_URL model_learn || echo "already cloned"
%cd model_learn
# Only what our code imports that may be missing; do NOT install the full
# requirements.txt on Colab (it clobbers Colab's curated environment).
!pip -q install datasets tokenizers
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Train `small` on the GPU

Notes on the choices below:
- `limit=200_000` stories (~47M tokens) keeps the tokenized stream inside Colab's
  ~12GB RAM. The full dataset as a Python list would OOM. This is plenty for
  coherent little stories from a 14M model.
- No `torch.set_default_device` — `train()` moves each batch to the model's device
  itself, so `get_batch`'s CPU RNG stays deterministic and nothing errors on GPU.
- If your session risks timing out, lower `max_steps` (e.g. 10000); record whatever
  you actually used in `DEVLOG.md`.


In [ ]:
import sys; sys.path.insert(0, "src")
import torch
from slm.config import SMALL
from slm.model import LlamaSLM
from slm.tokenizer import train_tokenizer
from slm.data import tokenize_texts, load_tinystories
from slm.train import TrainConfig, train, plot_loss

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun."

texts = load_tinystories("train", limit=200_000)   # RAM-safe subset (streams)
print(f"{len(texts):,} stories")
tok = train_tokenizer(texts, vocab_size=SMALL.vocab_size,
                      save_path="checkpoints/small_tok.json")
data = tokenize_texts(tok, texts)
print(f"token stream: {len(data):,} tokens")

model = LlamaSLM(SMALL).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

cfg = TrainConfig(lr=6e-4, warmup_steps=200, max_steps=20000, batch_size=64,
                  context_len=SMALL.context_len, log_every=100, sample_every=1000,
                  ckpt_path="checkpoints/small.pt")
train(model, data, cfg, tok=tok)
plot_loss("checkpoints/small_loss.csv", "checkpoints/small_loss.png")

## 3. Quick in-Colab sanity check (optional)

Generate a story right here before downloading, to confirm the checkpoint is good.


In [ ]:
from slm.sample import generate_text
print(generate_text("checkpoints/small.pt", "checkpoints/small_tok.json",
                    prompt="Once upon a time", max_new_tokens=120))

## 4. Download the artifacts

Save all three into your local repo's `checkpoints/` directory.


In [ ]:
from google.colab import files
files.download("checkpoints/small.pt")
files.download("checkpoints/small_tok.json")
files.download("checkpoints/small_loss.png")